# XGBoost Ablation: Current Final Set Plus `CN112313335A` Hotspots

This notebook starts from the current final XGBoost focused ablation and additionally removes the two strongest remaining `CN112313335A` concentration hotspots.

In [1]:
%load_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

project_root = Path.cwd().resolve()
while not (project_root / "utils").exists() and project_root != project_root.parent:
    project_root = project_root.parent

if not (project_root / "utils").exists():
    raise RuntimeError("Could not locate project root containing the utils package")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from utils.merge_historic_data import load_merged_dataset
from utils.pipeline import SiRNADataPipeline
from utils.splitter import GroupKFoldLeakPerGroup

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)


## Build Current Pipeline Data

This notebook uses the same strict-cleaning, add-mRNA, no-normalized-conditions pipeline and the same frozen by-gene+mRNA XGBoost hyperparameters as the other investigation notebooks.

In [2]:
cmsirna_path = os.environ.get("CMSIRNA_RAW_DATA_PATH")
historic_path = os.environ.get("CMSIRNA_RAW_HISTORIC_DATA_PATH")

assert cmsirna_path, "CMSIRNA_RAW_DATA_PATH is not set"
assert historic_path, "CMSIRNA_RAW_HISTORIC_DATA_PATH is not set"

raw_df = load_merged_dataset(cmsirna_path, historic_path)
pipeline = SiRNADataPipeline(target_len=25, fetch_missing_mrna=True)
enriched_df = pipeline.enrich_dataset_with_encodings(
    raw_df,
    strict_cleaning=True,
    add_mrna=True,
)

enriched_df["patent_group"] = enriched_df.get(
    "patent_ID", pd.Series(index=enriched_df.index, dtype=object)
).fillna("HISTORIC_OR_UNKNOWN")
enriched_df["Authorization_status"] = enriched_df.get(
    "Authorization_status", pd.Series(index=enriched_df.index, dtype=object)
).fillna("UNKNOWN")
enriched_df["Title"] = enriched_df.get(
    "Title", pd.Series(index=enriched_df.index, dtype=object)
).fillna("HISTORIC_OR_UNKNOWN")


loaded 3515 historic rows
merged 43153 CMsiRNA and 3515 historic rows into 46668
Running qc and data cleaning
dropped 4233 rows for in-vivo readings
dropped 565 rows for mM readings
dropped 115 rows for unknown or unwanted cell lines
dropped 47 rows for out-of-range inhibition
dropped 1749 rows for missing or unknown concentration
dropped 796 rows for concentration > 200 nM
filled 7522 rows for missing time of administration
dropped 2198 rows with a missing or >25 nt strand
dropped 6 columns: ['Modification_locations_Sense_strand', 'Modification_locations_Antisense_strand', 'Modifications_sense_strand', 'Modifications_AntiSense_strand_3_5', 'position_Antisense_strand', 'position_Sense_strand']
Mapping mRNA structural profiles
Error reading reference dataset: [Errno 2] No such file or directory: '/home/larsena8/software/fennec/src/fennec/support_files/train_data_v1.1.0_N=27742.csv'
Loaded 47 gene sequences from local cache
Loaded 0 gene sequences from reference CSV
Building gene -> mRNA

## What This Notebook Adds On Top Of The Current Final Ablation

This notebook keeps the current 5-hotspot XGBoost ablation and additionally removes:
- `CN112313335A @ 0.1 nM`
- `CN112313335A @ 10.0 nM`

Why these two were added:
- `CN112313335A @ 0.1 nM`: `n_samples = 184`, `spearman = -0.467330`, `mae = 87.533814`, `mean_true = 98.657065`, `mean_pred = 11.123251`
- `CN112313335A @ 10.0 nM`: `n_samples = 182`, `spearman = -0.469431`, `mae = 54.133396`, `mean_true = 92.877308`, `mean_pred = 38.743912`

These were the strongest remaining high-sample `Concentration + Patent` warning slices after the current final focused ablation, so this notebook tests whether removing them gives one last meaningful gain or whether the current final 5-hotspot set is already enough.


In [3]:
combo_mask = (
    ((enriched_df["gene_target_symbol_name"] == "APP") & (np.isclose(enriched_df["Concentration_nM"], 20.0, equal_nan=False)))
    | ((enriched_df["gene_target_symbol_name"] == "INHBE") & (np.isclose(enriched_df["Concentration_nM"], 100.0, equal_nan=False)))
    | ((enriched_df["gene_target_symbol_name"] == "PCSK9") & (np.isclose(enriched_df["Concentration_nM"], 100.0, equal_nan=False)))
    | ((enriched_df["patent_group"] == "US20220380773A1") & (np.isclose(enriched_df["Concentration_nM"], 20.0, equal_nan=False)))
    | (enriched_df["Cell_Type"] == "Primary mouse hepatocytes")
    | ((enriched_df["patent_group"] == "CN112313335A") & (np.isclose(enriched_df["Concentration_nM"], 0.1, equal_nan=False)))
    | ((enriched_df["patent_group"] == "CN112313335A") & (np.isclose(enriched_df["Concentration_nM"], 10.0, equal_nan=False)))
)
removed_df = enriched_df.loc[combo_mask].copy()
filtered_df = enriched_df.loc[~combo_mask].reset_index(drop=True)

removal_summary = pd.DataFrame([{
    "starting_rows": int(len(enriched_df)),
    "rows_removed": int(len(removed_df)),
    "rows_remaining": int(len(filtered_df)),
    "pct_removed": float(len(removed_df) / len(enriched_df)),
    "removed_hotspots": "APP@20, INHBE@100, PCSK9@100, US20220380773A1@20, Primary mouse hepatocytes, CN112313335A@0.1, CN112313335A@10",
}])

X, groups, y = pipeline.prepare_for_classical_ml(
    filtered_df,
    target_column="Inhibition",
    use_normalized_conditions=False,
)
mask = ~np.isnan(y)
X = X[mask]
groups = groups[mask]
y = y[mask]
analysis_df = filtered_df.loc[mask].reset_index(drop=True).copy()

print("Filtered dataframe shape:", analysis_df.shape)
print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
print("Unique genes:", len(np.unique(groups)))
removal_summary


Feature matrix X shape: (34117, 1425), target y shape: (34117,)
Filtered dataframe shape: (34117, 40)
Feature matrix shape: (34117, 1425)
Target shape: (34117,)
Unique genes: 54


,starting_rows,rows_removed,rows_remaining,pct_removed,removed_hotspots
0,35444,1327,34117,0.037439,"APP@20, INHBE@100, PCSK9@100, US20220380773A1@..."


## Helper Functions

In [4]:
def make_model(frozen_params):
    return XGBRegressor(
        tree_method="hist",
        n_jobs=-1,
        random_state=42,
        **frozen_params,
    )


def evaluate(y_true, y_pred):
    return {
        "spearman": float(spearmanr(y_true, y_pred).statistic),
        "pearson": float(pearsonr(y_true, y_pred).statistic),
        "rmse": float(mean_squared_error(y_true, y_pred) ** 0.5),
        "mae": float(mean_absolute_error(y_true, y_pred)),
    }


def grouped_spearman(df, group_cols, min_samples=10):
    rows = []
    for keys, group_df in df.groupby(group_cols, dropna=False):
        if len(group_df) < min_samples:
            continue
        corr = spearmanr(group_df["y_true"], group_df["y_pred"], nan_policy="omit").statistic
        if pd.isna(corr):
            continue
        if not isinstance(keys, tuple):
            keys = (keys,)
        row = {col: value for col, value in zip(group_cols, keys)}
        row.update({
            "n_samples": len(group_df),
            "spearman": float(corr),
            "mae": float(group_df["abs_error"].mean()),
            "mean_true": float(group_df["y_true"].mean()),
            "mean_pred": float(group_df["y_pred"].mean()),
        })
        rows.append(row)
    return pd.DataFrame(rows).sort_values(["spearman", "mae"], ascending=[True, False]).reset_index(drop=True)


def issue_summary(df, group_col, min_samples=20):
    rows = []
    for value, group_df in df.groupby(group_col, dropna=False):
        if len(group_df) < min_samples:
            continue
        corr = spearmanr(group_df["y_true"], group_df["y_pred"], nan_policy="omit").statistic
        rows.append({
            group_col: value,
            "n_samples": len(group_df),
            "mae": float(group_df["abs_error"].mean()),
            "rmse": float(np.sqrt(np.mean(np.square(group_df["residual"])))),
            "spearman": float(corr) if not pd.isna(corr) else np.nan,
            "mean_true": float(group_df["y_true"].mean()),
            "mean_pred": float(group_df["y_pred"].mean()),
        })
    return pd.DataFrame(rows).sort_values(["spearman", "mae"], ascending=[True, False]).reset_index(drop=True)


## Refit And Re-Analyze

The model is retrained from scratch after adding the two `CN112313335A` removals on top of the existing final focused ablation set.

In [ ]:
frozen_params = {'n_estimators': 800, 'max_depth': 4, 'learning_rate': 0.15881823130907038, 'subsample': 0.8812898741586134, 'colsample_bytree': 0.7824379872752019, 'min_child_weight': 4, 'reg_lambda': 0.8342807691178866, 'reg_alpha': 1.4296995092035882, 'gamma': 0.07531958697602548}
baseline_metrics = pd.DataFrame([{'spearman': 0.45886, 'pearson': 0.453422, 'rmse': 31.421689, 'mae': 24.568816}])
current_final_metrics = pd.DataFrame([{'spearman': 0.493612, 'pearson': 0.491417, 'rmse': 30.431602, 'mae': 23.661806}])
gene_cv = GroupKFoldLeakPerGroup(n_splits=3, leak_n=0, random_state=42)

fold_rows = []
oof_frames = []
oof_true = []
oof_pred = []

for fold_id, (train_idx, test_idx) in enumerate(gene_cv.split(X, y, groups), start=1):
    model = make_model(frozen_params)
    model.fit(X[train_idx], y[train_idx])
    fold_pred = model.predict(X[test_idx])

    oof_true.append(y[test_idx])
    oof_pred.append(fold_pred)

    fold_frame = analysis_df.iloc[test_idx].reset_index().rename(columns={"index": "source_index"}).copy()
    fold_frame["row_index"] = test_idx
    fold_frame["fold_id"] = fold_id
    fold_frame["group"] = groups[test_idx]
    fold_frame["y_true"] = y[test_idx]
    fold_frame["y_pred"] = fold_pred
    fold_frame["residual"] = fold_frame["y_true"] - fold_frame["y_pred"]
    fold_frame["abs_error"] = fold_frame["residual"].abs()
    oof_frames.append(fold_frame)

    fold_rows.append({
        "fold_id": fold_id,
        "n_train": int(len(train_idx)),
        "n_test": int(len(test_idx)),
        "n_train_groups": int(len(np.unique(groups[train_idx]))),
        "n_test_groups": int(len(np.unique(groups[test_idx]))),
    })

predictions_df = pd.concat(oof_frames, ignore_index=True)
metrics_df = pd.DataFrame([evaluate(np.concatenate(oof_true), np.concatenate(oof_pred))])
fold_summary = pd.DataFrame(fold_rows)
comparison = pd.concat([
    baseline_metrics.assign(run="baseline_by_gene_mrna"),
    current_final_metrics.assign(run="current_final_5_hotspots"),
    metrics_df.assign(run="final_plus_cn112313335a"),
], ignore_index=True)
delta_vs_baseline = metrics_df.iloc[0] - baseline_metrics.iloc[0]
delta_vs_current_final = metrics_df.iloc[0] - current_final_metrics.iloc[0]
delta_df = pd.DataFrame({
    "metric": delta_vs_baseline.index,
    "delta_vs_baseline": delta_vs_baseline.values,
    "delta_vs_current_final": delta_vs_current_final.values,
})
fold_summary


,fold_id,n_train,n_test,n_train_groups,n_test_groups
0,1,23217,10900,54,15
1,2,23249,10868,54,16
2,3,23282,10835,54,17


## Fold Summary

## Comparison Against Previous Setups

In [6]:
comparison

,spearman,pearson,rmse,mae,run
0,0.458860,0.453422,31.421689,24.568816,baseline_by_gene_mrna
1,0.493612,0.491417,30.431602,23.661806,current_final_5_hotspots
2,0.498824,0.497458,29.921500,23.525860,final_plus_cn112313335a


## Metric Delta

In [7]:
delta_df

,metric,delta_vs_baseline,delta_vs_current_final
0,spearman,0.039964,0.005212
1,pearson,0.044036,0.006041
2,rmse,-1.500189,-0.510102
3,mae,-1.042956,-0.135946


## Overall Metrics

In [8]:
metrics_df

,spearman,pearson,rmse,mae
0,0.498824,0.497458,29.9215,23.52586


## Concentration Ranking Slice

These tables show where XGBoost preserves ranking poorly or well across concentration alone and the key concentration combinations after adding the two `CN112313335A` hotspot removals.

In [9]:
spearman_by_concentration = grouped_spearman(predictions_df, ["Concentration_nM"], min_samples=20)
spearman_by_concentration_gene = grouped_spearman(predictions_df, ["Concentration_nM", "group"], min_samples=10)
spearman_by_concentration_patent = grouped_spearman(predictions_df, ["Concentration_nM", "patent_group"], min_samples=10)

spearman_by_concentration.sort_values(["spearman", "mae"], ascending=[True, False]).head(20)

,Concentration_nM,n_samples,spearman,mae,mean_true,mean_pred
0,0.00030,40,-0.242916,13.983462,5.699000,-4.360168
1,0.00300,64,-0.226371,16.113919,3.584531,-0.383828
2,6.66670,49,-0.143579,14.403276,78.425510,76.731728
3,0.02740,49,-0.110278,25.956698,30.445306,32.195072
4,2.22220,48,-0.084471,14.506299,78.288958,76.235405
5,0.01600,112,-0.057892,19.818027,21.753125,20.164366
6,0.00910,49,-0.049448,19.973818,12.247959,14.437309
7,0.00100,64,-0.048930,19.048380,8.084375,-4.422038
8,0.00320,113,-0.025710,13.485571,1.700531,5.930970
9,0.08230,48,-0.010532,23.464343,51.607917,50.603333


## Concentration + Gene Hotspots

In [10]:
spearman_by_concentration_gene.sort_values(["spearman", "mae"], ascending=[True, False]).head(20)

,Concentration_nM,group,n_samples,spearman,mae,mean_true,mean_pred
0,0.0030,HSD17B13,23,-0.558300,19.509716,9.405217,-9.504380
1,0.0800,HSD17B13,13,-0.450000,23.611899,72.375385,48.763485
2,5.0000,AGT,26,-0.387417,31.378396,70.858077,40.369236
3,0.0270,HSD17B13,16,-0.341176,39.306344,42.193750,3.068857
4,0.4000,HSD17B13,13,-0.338889,19.512433,84.767692,65.255257
5,2.2220,HSD17B13,16,-0.329412,27.395988,87.950000,63.383900
6,0.0010,AGT,20,-0.317874,13.260709,-0.747500,-1.420854
7,6.6670,HSD17B13,16,-0.317647,28.469449,88.043750,62.019012
8,0.0090,HSD17B13,15,-0.285714,28.742527,23.246667,-5.495860
9,0.0003,HSD17B13,21,-0.280052,12.878266,4.458095,-4.655667


## Concentration + Patent Hotspots

In [11]:
spearman_by_concentration_patent.sort_values(["spearman", "mae"], ascending=[True, False]).head(20)

,Concentration_nM,patent_group,n_samples,spearman,mae,mean_true,mean_pred
0,0.0030,CN116940681A,16,-0.488235,19.925633,9.793750,-9.269215
1,0.1000,CN116940681A,21,-0.430659,41.005527,86.514286,46.073639
2,5.0000,CN117070517A,26,-0.387417,31.378396,70.858077,40.369236
3,10.0000,CN117070517A,16,-0.355882,32.337818,65.556875,33.219055
4,0.0270,CN116940681A,16,-0.341176,39.306344,42.193750,3.068857
5,2.2220,CN116940681A,16,-0.329412,27.395988,87.950000,63.383900
6,0.0010,WO2023088427A1,20,-0.317874,13.260709,-0.747500,-1.420854
7,6.6670,CN116940681A,16,-0.317647,28.469449,88.043750,62.019012
8,0.0090,CN116940681A,15,-0.285714,28.742527,23.246667,-5.495860
9,0.0003,CN116940681A,16,-0.257732,12.726007,5.756250,-4.531595


## Experimental Feature Issues

In [12]:
issue_by_cell_type = issue_summary(predictions_df, "Cell_Type", min_samples=30)
issue_by_concentration = issue_summary(predictions_df, "Concentration_nM", min_samples=20)
issue_by_time = issue_summary(predictions_df, "Time_of_administration_h", min_samples=20)
issue_by_patent = issue_summary(predictions_df, "patent_group", min_samples=20)
issue_by_authorization = issue_summary(predictions_df, "Authorization_status", min_samples=20)

issue_by_cell_type.sort_values(["spearman", "mae"], ascending=[True, False]).head(20)

,Cell_Type,n_samples,mae,rmse,spearman,mean_true,mean_pred
0,Non-human hepatocytes,93,20.888543,27.214448,0.006222,17.944086,34.824261
1,Primary human hepatocytes,2839,26.703178,32.722192,0.073352,51.060032,50.078053
2,Primary Cynomolgus Monkey Hepatocytes,4140,24.559549,31.610419,0.326881,33.508838,37.373375
3,HepG2,1528,21.702642,29.052409,0.386840,39.723835,48.132637
4,Hep3B,11425,26.764033,33.354193,0.435022,40.445743,39.005268
5,Huh7,1544,22.588908,28.574411,0.495724,40.916937,38.550922
6,HEK293 Cells,139,18.456558,23.530472,0.506326,70.431589,64.865730
7,Human iPSC-derived cortical neurons,199,23.981305,27.440000,0.523541,42.732663,45.646126
8,COS7,3747,20.822004,26.003849,0.548590,24.770571,32.189220
9,Hela,1643,22.277433,27.362886,0.578338,39.518917,39.898857


## Concentration Issue Summary

In [13]:
issue_by_concentration.sort_values(["spearman", "mae"], ascending=[True, False]).head(20)

,Concentration_nM,n_samples,mae,rmse,spearman,mean_true,mean_pred
0,0.00030,40,13.983462,17.267820,-0.242916,5.699000,-4.360168
1,0.00300,64,16.113919,21.281572,-0.226371,3.584531,-0.383828
2,6.66670,49,14.403276,18.528683,-0.143579,78.425510,76.731728
3,0.02740,49,25.956698,30.506211,-0.110278,30.445306,32.195072
4,2.22220,48,14.506299,17.868121,-0.084471,78.288958,76.235405
5,0.01600,112,19.818027,24.768233,-0.057892,21.753125,20.164366
6,0.00910,49,19.973818,25.033128,-0.049448,12.247959,14.437309
7,0.00100,64,19.048380,23.398782,-0.048930,8.084375,-4.422038
8,0.00320,113,13.485571,18.343036,-0.025710,1.700531,5.930970
9,0.08230,48,23.464343,26.653244,-0.010532,51.607917,50.603333


## Time Issue Summary

In [14]:
issue_by_time.sort_values(["spearman", "mae"], ascending=[True, False]).head(20)

,Time_of_administration_h,n_samples,mae,rmse,spearman,mean_true,mean_pred
0,40.0,197,20.532066,24.078398,0.339198,27.381218,18.595999
1,72.0,1385,21.338758,26.310396,0.426293,15.762202,29.993355
2,24.0,22944,25.172959,31.767000,0.451783,40.868692,41.290443
3,168.0,199,23.981305,27.440000,0.523541,42.732663,45.646126
4,48.0,7862,19.182639,24.724359,0.578657,48.647754,45.599144


## Patent Issue Summary

In [15]:
issue_by_patent.sort_values(["spearman", "mae"], ascending=[True, False]).head(20)

,patent_group,n_samples,mae,rmse,spearman,mean_true,mean_pred
0,WO2023091644A2,1434,27.923160,34.220422,-0.161155,48.087301,53.169460
1,CN116801886A,263,35.248812,41.255260,-0.087584,50.889696,41.111713
2,TW202321444A,93,20.888543,27.214448,0.006222,17.944086,34.824261
3,WO2022121959A1,100,31.279891,42.070285,0.010527,64.982500,65.231682
4,CN117070517A,88,27.560368,32.332017,0.017065,65.132727,38.686405
5,WO2023213284A1,48,35.475281,37.484918,0.074426,86.482083,51.692387
6,CN117448322A,132,46.574461,52.798349,0.085268,45.159848,11.191084
7,CN117210468A,224,22.553953,26.375137,0.101176,25.340179,22.110626
8,CN117106781A,20,30.187527,34.395734,0.118797,65.312500,35.124969
9,WO2023056446A1,222,23.565608,30.265853,0.124683,61.045045,51.529068


## Authorization Issue Summary

In [16]:
issue_by_authorization.sort_values(["spearman", "mae"], ascending=[True, False]).head(20)

,Authorization_status,n_samples,mae,rmse,spearman,mean_true,mean_pred
0,Published,224,22.553953,26.375137,0.101176,25.340179,22.110626
1,Withdrawn,1081,24.353075,29.580902,0.434309,30.121092,33.882191
2,Unknown Status,12203,23.037386,29.632403,0.499973,33.763198,44.524277
3,Substantive Examination,15143,25.556965,31.871753,0.521720,44.716137,36.011543
4,Granted,1619,23.011086,28.991401,0.541787,48.671711,51.216183
5,UNKNOWN,2333,12.964667,16.422984,0.650786,64.357525,63.261734


## Interpretation Notes

- This notebook tests whether adding the two strongest remaining `CN112313335A` hotspot removals on top of the current final 5-hotspot set gives one more meaningful gain.
- The most important comparison is not only against the original by-gene baseline, but also against the current final 5-hotspot notebook.
- If this notebook improves clearly over the current final 5-hotspot notebook, then `CN112313335A @ 0.1 nM` and `10.0 nM` are likely worth including in the final pre-CNN filter.
- If the gain is marginal or unstable, then the current final 5-hotspot set is probably already the better stopping point.
- The remaining post-ablation weakest slices should show whether `CN112313335A` was really the next key driver or whether the residual problem has already shifted mainly toward target-specific biology such as `HSD17B13`.

## Conclusions And Next Step

- Compare this notebook directly against the current final 5-hotspot notebook.
- If the metric gains over `current_final_5_hotspots` are small, keep the leaner current final set.
- If the gains are meaningful, this expanded set becomes the new XGBoost winner.
- After that comparison, I would stop the ablation loop and move to the CNN pipeline with the better of the two filters.
